In [0]:
%sql
-- Carga: lee los 8 CSV del volumen y los guarda como tablas
CREATE OR REPLACE TABLE workspace.electiva.customers AS SELECT * FROM read_files('/Volumes/workspace/default/mi_volumen_datos/customers.csv', format => 'csv', header => true);
CREATE OR REPLACE TABLE workspace.electiva.employee AS SELECT * FROM read_files('/Volumes/workspace/default/mi_volumen_datos/employees.csv', format => 'csv', header => true);
CREATE OR REPLACE TABLE workspace.electiva.orders AS SELECT * FROM read_files('/Volumes/workspace/default/mi_volumen_datos/orders.csv', format => 'csv', header => true);
CREATE OR REPLACE TABLE workspace.electiva.product_categories AS SELECT * FROM read_files('/Volumes/workspace/default/mi_volumen_datos/productcategories.csv', format => 'csv', header => true);
CREATE OR REPLACE TABLE workspace.electiva.product_sub_categories AS SELECT * FROM read_files('/Volumes/workspace/default/mi_volumen_datos/productsubcategories.csv', format => 'csv', header => true);
CREATE OR REPLACE TABLE workspace.electiva.products AS SELECT * FROM read_files('/Volumes/workspace/default/mi_volumen_datos/products.csv', format => 'csv', header => true);
CREATE OR REPLACE TABLE workspace.electiva.vendor AS SELECT * FROM read_files('/Volumes/workspace/default/mi_volumen_datos/vendors.csv', format => 'csv', header => true);
CREATE OR REPLACE TABLE workspace.electiva.vendor_product AS SELECT * FROM read_files('/Volumes/workspace/default/mi_volumen_datos/vendorproduct.csv', format => 'csv', header => true);

-- Punto 1: arma el user_name (id + 2 letras del nombre + 2 del apellido + AW) y lo muestra con el nombre completo
SELECT customerid, fullname,
       CONCAT_WS('-', customerid, UPPER(LEFT(firstname, 2)), LOWER(LEFT(lastname, 2)), 'AW') AS user_name
FROM workspace.electiva.customers;

-- Punto 2: crea la tabla page_user y la llena con el id y el user_name de cada cliente
CREATE OR REPLACE TABLE workspace.electiva.page_user (customerid INT, user_name STRING);

INSERT INTO workspace.electiva.page_user
SELECT customerid, CONCAT_WS('-', customerid, UPPER(LEFT(firstname, 2)), LOWER(LEFT(lastname, 2)), 'AW')
FROM workspace.electiva.customers;

SELECT * FROM workspace.electiva.page_user;

-- Punto 3: lista las compras de agosto 2011 con su descuento según la categoría y el total ya descontado
CREATE OR REPLACE VIEW workspace.electiva.v_ordenes_mitad_anio AS
SELECT prod.productname,
       prod.modelname,
       usr.user_name,
       ord.salesorderid,
       cat.categoryname,
       TO_DATE(ord.orderdate, 'M/d/yyyy') AS orderdate,
       ord.linetotal,
       COALESCE(dto.descuento, 0) AS `descuento_mitad_año`,
       ord.linetotal - ord.linetotal * COALESCE(dto.descuento, 0) AS total_descuentos_aplicados
FROM workspace.electiva.orders ord
JOIN workspace.electiva.page_user usr ON ord.customerid = usr.customerid
JOIN workspace.electiva.products prod ON ord.productid = prod.productid
JOIN workspace.electiva.product_sub_categories sub ON prod.subcategoryid = sub.subcategoryid
JOIN workspace.electiva.product_categories cat ON sub.categoryid = cat.categoryid
LEFT JOIN (VALUES ('Bikes', 0.10), ('Clothing', 0.15), ('Components', 0.08), ('Accessories', 0.20)) AS dto(categoria, descuento)
       ON cat.categoryname = dto.categoria
WHERE TO_DATE(ord.orderdate, 'M/d/yyyy') BETWEEN '2011-08-01' AND '2011-08-31';

SELECT * FROM workspace.electiva.v_ordenes_mitad_anio;

-- Punto 4: cuenta las órdenes distintas de cada cliente en 2011 y lo clasifica como Oro, Plata u Ocasional
CREATE OR REPLACE MATERIALIZED VIEW workspace.electiva.customer_class AS
SELECT amount_of_orders,
       user_name,
       customerfullname,
       CASE WHEN amount_of_orders >= 3 THEN 'Oro'
            WHEN amount_of_orders = 2 THEN 'Plata'
            ELSE 'Ocasional' END AS user_level,
       customerid
FROM (
    SELECT usr.customerid,
           usr.user_name,
           cli.fullname AS customerfullname,
           COUNT(DISTINCT ord.salesorderid) AS amount_of_orders
    FROM workspace.electiva.orders ord
    JOIN workspace.electiva.page_user usr ON ord.customerid = usr.customerid
    JOIN workspace.electiva.customers cli ON ord.customerid = cli.customerid
    WHERE TO_DATE(ord.orderdate, 'M/d/yyyy') BETWEEN '2011-01-01' AND '2011-12-31'
    GROUP BY usr.customerid, usr.user_name, cli.fullname
);

SELECT * FROM workspace.electiva.customer_class;